# Orchestrator Test

Smoke test for `ModelsOrchestrator` — data loading, MLP training, LightGBM training, and combined prediction.

In [1]:
import sys                                                                                                                                                                                                
print(sys.executable)

c:\Users\angej\Documents\2_Programação\health_index_project\.venv\Scripts\python.exe


In [2]:
import sys
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from models_classes.models_orchestrator import ModelsOrchestrator
from models_classes.mlp_disease_neural_net import device

print(f'PyTorch: {torch.__version__}')
print(f'Using device: {device}')

Python: c:\Users\angej\Documents\2_Programação\health_index_project\.venv\Scripts\python.exe
PyTorch: 2.10.0+cu128
CUDA built with: 12.8
CUDA available: True
Using device: cuda
PyTorch: 2.10.0+cu128
Using device: cuda


c:\Users\angej\Documents\2_Programação\health_index_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load data

In [3]:
orchestrator = ModelsOrchestrator(type_disease='chikungunya')

x_train_cat, x_test_cat, x_train_num, x_test_num, y_train, y_test, embedding_sizes = orchestrator.prepare_data()
numerical_columns = orchestrator.df.drop(columns=list(orchestrator.categorical_columns) + ['final_classification']).columns

print(f'Train: {len(y_train)} | Test: {len(y_test)}')
print(f'Categorical features: {len(orchestrator.categorical_columns)}')
print(f'Numerical features:   {len(numerical_columns)}')

Colunas removidas (>99% mesmo valor): 13
['blood_disorder', 'liver_disease', 'kidney_disease', 'peptic_ulcer', 'autoimmune_disease', 'nosebleed', 'gum_bleeding', 'metrorrhagia', 'petechiae_hemorrh', 'hematuria', 'other_bleeding', 'symptom_year_end', 'hemorrhagic_count']
Train: 433039 | Test: 48116
Categorical features: 9
Numerical features:   86


## 2. Train MLP

In [4]:
mlp_model = orchestrator.train_mlp(epochs = 10, embedding_sizes=embedding_sizes,save_path='C:\\Users\\angej\\Documents\\2_Programação\\health_index_project\\models_saved\\best_orchestrator_mlp.pth')

Epoch   0 | Train: 0.3508 | Val: 0.3097 | LR: 0.000100
Epoch   1 | Train: 0.3237 | Val: 0.2996 | LR: 0.000100
Epoch   2 | Train: 0.3149 | Val: 0.2940 | LR: 0.000100
Epoch   3 | Train: 0.3086 | Val: 0.2901 | LR: 0.000100
Epoch   4 | Train: 0.3045 | Val: 0.2873 | LR: 0.000100
Epoch   5 | Train: 0.3006 | Val: 0.2849 | LR: 0.000100
Epoch   6 | Train: 0.2981 | Val: 0.2831 | LR: 0.000100
Epoch   7 | Train: 0.2953 | Val: 0.2820 | LR: 0.000100
Epoch   8 | Train: 0.2931 | Val: 0.2804 | LR: 0.000100
Epoch   9 | Train: 0.2911 | Val: 0.2796 | LR: 0.000100


### 2.1 MLP evaluation

In [5]:
display(mlp_model.evaluate(orchestrator.test_loader, orchestrator.y_test))

 Threshold |   Accuracy |  Precision |     Recall |         F1
------------------------------------------------------------
      0.30 |     0.8124 |     0.8274 |     0.9331 |     0.8771
      0.35 |     0.8108 |     0.8402 |     0.9091 |     0.8733
      0.40 |     0.8042 |     0.8518 |     0.8801 |     0.8657
      0.45 |     0.7929 |     0.8640 |     0.8441 |     0.8539
      0.50 |     0.7764 |     0.8757 |     0.8021 |     0.8373
      0.55 |     0.7562 |     0.8878 |     0.7555 |     0.8163
      0.60 |     0.7315 |     0.9009 |     0.7030 |     0.7897


,Actual,Prob,Predicted,Correct
0,1,0.752995,1,True
1,1,0.278652,0,False
2,1,0.983374,1,True
3,1,0.513701,1,True
4,1,0.829170,1,True
...,...,...,...,...
48111,0,0.211100,0,True
48112,0,0.190366,0,True
48113,1,0.878040,1,True
48114,1,0.805761,1,True


## 3. Train LightGBM

In [6]:
lgbm_model = orchestrator.train_lgbm(fast_train=True, x_train_cat=x_train_cat, x_train_num=x_train_num, y_train=y_train)
lgbm_model.evaluate(x_test_cat, x_test_num, y_test, orchestrator.categorical_columns, orchestrator.numerical_columns)

[LightGBM] [Info] Number of positive: 309978, number of negative: 123061
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 1061
[LightGBM] [Info] Number of data points in the train set: 433039, number of used features: 95
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 4060 Ti, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 18 dense feature groups (8.26 MB) transferred to GPU in 0.011553 secs. 1 sparse feature groups
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.715820 -> initscore=0.923821
[LightGBM] [Info] Start training from score 0.923821
 Threshold |   Accuracy |  Precision |     Recall |         F1
------------------------------------------------------------
      0.10 |     0.7632 |     0.7530 |     0.9968 |     0.8579
      0.30 |     0.8198 |     0.8097 |     0.9789 |     0.8863
      0.35 |  

## 3. Train XGB

In [7]:
xgb_model = orchestrator.train_xgb(fast_train=True, x_train_cat=x_train_cat, x_train_num=x_train_num, y_train=y_train)
xgb_model.evaluate(x_test_cat, x_test_num, y_test, orchestrator.categorical_columns, orchestrator.numerical_columns)

c:\Users\angej\Documents\2_Programação\health_index_project\.venv\Lib\site-packages\xgboost\core.py:751: UserWarning: [20:51:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


 Threshold |   Accuracy |  Precision |     Recall |         F1
------------------------------------------------------------
      0.10 |     0.7674 |     0.7566 |     0.9963 |     0.8600
      0.30 |     0.8200 |     0.8107 |     0.9771 |     0.8862
      0.35 |     0.8281 |     0.8228 |     0.9691 |     0.8899
      0.40 |     0.8339 |     0.8335 |     0.9601 |     0.8923
      0.45 |     0.8384 |     0.8450 |     0.9487 |     0.8939
      0.50 |     0.8403 |     0.8551 |     0.9358 |     0.8936
      0.55 |     0.8414 |     0.8670 |     0.9200 |     0.8927
      0.60 |     0.8382 |     0.8782 |     0.8992 |     0.8885


## 4. Combined prediction (MLP + LightGBM average)

In [8]:
confirmation_df = orchestrator.evaluate_combined(threshold=0.4, mlp_model=mlp_model, lgbm_model=lgbm_model, xgb_model=xgb_model, x_test_cat=x_test_cat, x_test_num=x_test_num)
display(confirmation_df['unanimous'].value_counts())

 Threshold |   Accuracy |  Precision |     Recall |         F1
------------------------------------------------------------
      0.30 |     0.8223 |     0.8146 |     0.9739 |     0.8872
      0.35 |     0.8298 |     0.8267 |     0.9650 |     0.8905
      0.40 |     0.8351 |     0.8386 |     0.9536 |     0.8924
      0.45 |     0.8384 |     0.8516 |     0.9381 |     0.8928
      0.50 |     0.8387 |     0.8646 |     0.9191 |     0.8910
      0.55 |     0.8340 |     0.8769 |     0.8941 |     0.8854
      0.60 |     0.8229 |     0.8900 |     0.8594 |     0.8744
MLP TP: 88.01%
LGBM TP: 96.31%
XGB TP: 96.01%


unanimous
True     34728
False    13388
Name: count, dtype: int64

In [ ]:
patient = {                                                                                                                                                                                                                    # Symptoms (1=Yes, 0=No)
      'fever': 1,                                                                                                                                                                                                          
      'headache': 1,                                                                                                                                                                                                       
      'myalgia': 1,                                                                                                                                                                                                        
      'joint_pain': 1,       
      'vomiting': 0,
      'nausea': 0,
      'rash': 1,             
      'back_pain': 1,
      'conjunctivitis': 0,
      'arthritis': 1,       
      'petechiae': 0,
      'retro_orbital_pain': 0,  

      # Demographics
      'age': 45,
      'sex': 'Feminino',
      'pregnancy_status': 'Não',
      'race': 'Parda',
      'education_level': 'Ensino médio completo',
      'occupation_code': 'DONA DE CASA',
      'residence_state': 'Bahia',       

      # Dates
      'symptom_onset_date': '2025-02-10',
      'notification_date': '2025-02-13',
  }

result = orchestrator.predict(patient, mlp_model, lgbm_model, xgb_model, threshold=0.4)
print(result)

{'mlp_probability': 0.6625786423683167, 'lgbm_probability': 0.826087611486943, 'xgb_probability': 0.7495610117912292, 'mlp_prediction': 1, 'lgbm_prediction': 1, 'xgb_prediction': 1, 'average_probability': 0.7460757552154963, 'weighted_probability': 0.7485440275881194, 'final_prediction': 1, 'unanimous': True}
